In [2]:
import numpy as np
import pandas as pd
import os
import torch
from torch.utils.data import Dataset, DataLoader
from itertools import product
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import warnings
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.utils import to_dense_adj, dense_to_sparse
from torch_geometric.nn.models import GCN
# warnings.filterwarnings('ignore')
from torch_geometric.nn import GATConv
from torch import Tensor
from torch_geometric.utils import scatter
from torch.linalg import vector_norm
from typing import Optional
import numpy as np
import torch
import torch.optim as optim
# import lightning as L
# from lightning.pytorch.callbacks import Callback, BatchSizeFinder
from torch_geometric.loader import DataLoader

In [3]:
df = pd.read_csv("e927ce742c884955bf2a667929d36b2ef41c572cd6e245fa86257ecc2f7be7bc")
df1 = pd.read_csv("85b5cb4eea5a4f259766f42a448e2c04a7499c43e1ae4cc28fbdee8e087e2385")

In [5]:
# turbines = range(1, 135)
# days = range(1, 246)
# timestamps = pd.date_range("00:00", "23:50", freq="10T").strftime("%H:%M")
# expected = pd.DataFrame(
#     list(product(turbines, days, timestamps)),
#     columns=["TurbID", "Day", "Tmstamp"]
# )
# df1["Tmstamp"] = pd.to_datetime(df1["Tmstamp"]).dt.strftime("%H:%M")
# merged = expected.merge(df1, on=["TurbID", "Day", "Tmstamp"], how="left", indicator=True)
# missing_rows = merged[merged["_merge"] == "left_only"].drop(columns=["_merge"])
# missing_columns = df1[df1.isnull().any(axis=1)]
# missing_metadata = {
#     "missing_rows": missing_rows,
#     "missing_columns": missing_columns
# }
# print("Missing Rows:")
# print(missing_metadata["missing_rows"])
# print("\nRows with Missing Columns:")
# print(missing_metadata["missing_columns"])

In [6]:
# missing_counts = missing_columns.groupby(['Day', 'Tmstamp']).size().reset_index(name='Count')
# pivot_table = missing_counts.pivot(index='Day', columns='Tmstamp', values='Count').fillna(0)
# plt.figure(figsize=(12, 8))
# sns.heatmap(
#     pivot_table,
#     cmap='coolwarm',
#     annot=False,
#     fmt='.0f',
#     cbar_kws={'label': 'Count of Missing Turbines'},
#     mask=(pivot_table == 0),
#     vmin=0,
#     vmax=20
# )
# plt.title('Missing Data Heatmap')
# plt.xlabel('Tmstamp')
# plt.ylabel('Day')
# plt.show()

In [4]:
merged_df = pd.merge(df1, df[['TurbID', 'x', 'y']], on='TurbID', how='left')
print(merged_df)
filtered_df1 = merged_df
filtered_df1['Minutes'] = pd.to_datetime(filtered_df1['Tmstamp'], format='%H:%M').dt.hour * 60 + pd.to_datetime(filtered_df1['Tmstamp'], format='%H:%M').dt.minute
print(filtered_df1)

         TurbID  Day Tmstamp  Wspd  Wdir   Etmp   Itmp    Ndir  Pab1  Pab2  \
0             1    1   00:00   NaN   NaN    NaN    NaN     NaN   NaN   NaN   
1             1    1   00:10  6.17 -3.99  30.73  41.80   25.92  1.00  1.00   
2             1    1   00:20  6.27 -2.18  30.60  41.63   20.91  1.00  1.00   
3             1    1   00:30  6.42 -0.73  30.52  41.52   20.91  1.00  1.00   
4             1    1   00:40  6.25  0.89  30.49  41.38   20.91  1.00  1.00   
...         ...  ...     ...   ...   ...    ...    ...     ...   ...   ...   
4727515     134  245   23:10  7.79  2.80  -0.07   3.95  216.51  6.03  6.03   
4727516     134  245   23:20  8.06  4.39   0.23   3.94  216.51  5.81  5.81   
4727517     134  245   23:30  8.08  2.28  -0.16   4.15  216.51  0.68  0.68   
4727518     134  245   23:40  8.46  0.80  -0.14   4.32  216.51  0.02  0.02   
4727519     134  245   23:50  8.68  0.52  -0.06   4.39  216.51  0.01  0.01   

         Pab3    Prtv     Patv          x           y  
0      

In [5]:
feature_cols = ['Wspd', 'Wdir', 'Etmp', 'Itmp', 'Ndir', 'Pab1', 'Pab2', 'Pab3', 'Prtv', 'Patv', 'x', 'y', 'Minutes', 'Day']
timesteps = 1

class WindTurbineDataset(Dataset):
    def __init__(self, df, target_col='Patv', normalize=False, fit_scaler=False, scaler_dict=None):
        self.sequence_length = timesteps
        self.target_col = target_col
        self.normalize = normalize
        self.df_clean = self._clean_data(df)
        self.sequences = self._create_sequences()
        if normalize:
            self.scaler_dict = self._setup_scalers(fit_scaler, scaler_dict)
            self.sequences = self._normalize_sequences(self.sequences)

    def _clean_data(self, df):
        df_clean = df.dropna()
        return df_clean

    def _create_sequences(self):
        sequences = []
        self.df_clean = self.df_clean.sort_values(['Day', 'Tmstamp', 'TurbID'])
        unique_days = sorted(self.df_clean['Day'].unique())
        num_turbines = self.df_clean['TurbID'].nunique()
        for day in unique_days:
            day_data = self.df_clean[self.df_clean['Day'] == day]
            timestamps = sorted(day_data['Tmstamp'].unique())
            for i in range(len(timestamps) - self.sequence_length):
                sequence_timestamps = timestamps[i:i + self.sequence_length + 1]
                valid_sequence = True
                sequence_data = []
                for ts in sequence_timestamps:
                    ts_data = day_data[day_data['Tmstamp'] == ts]
                    if len(ts_data) != num_turbines:
                        valid_sequence = False
                        break
                    sequence_data.append(ts_data)
                if valid_sequence:
                    sequence_tensor = self._sequence_to_tensor(sequence_data)
                    sequences.append(sequence_tensor)
        return sequences

    def _sequence_to_tensor(self, sequence_data):
        input_data = []
        for i in range(self.sequence_length):
            frame_data = sequence_data[i].sort_values('TurbID')
            features = []
            for col in feature_cols:
                if col in frame_data.columns:
                    features.append(frame_data[col].values)
            frame_features = np.column_stack(features)
            input_data.append(frame_features)
        target_data = sequence_data[self.sequence_length].sort_values('TurbID')
        target = target_data[self.target_col].values
        input_tensor = torch.FloatTensor(np.array(input_data))
        target_tensor = torch.FloatTensor(target)
        return (input_tensor, target_tensor)

    def _setup_scalers(self, fit_scaler, scaler_dict):
        if scaler_dict and not fit_scaler:
            return scaler_dict
        all_data = {col: [] for col in feature_cols}
        for input_seq, _ in self.sequences:
            for feature_idx, col in enumerate(feature_cols):
                if feature_idx < input_seq.shape[2]:
                    all_data[col].extend(input_seq[:, :, feature_idx].flatten().numpy())
        scaler_dict = {}
        for col in feature_cols:
            if all_data[col]:
                scaler = StandardScaler()
                if fit_scaler:
                    scaler.fit(np.array(all_data[col]).reshape(-1, 1))
                scaler_dict[col] = scaler
        return scaler_dict

    def _normalize_sequences(self, sequences):
        normalized_sequences = []
        for input_seq, target in sequences:
            normalized_input = input_seq.clone()
            for feature_idx, col in enumerate(feature_cols):
                if col in self.scaler_dict and feature_idx < input_seq.shape[2]:
                    scaler = self.scaler_dict[col]
                    feature_data = input_seq[:, :, feature_idx].numpy()
                    normalized_feature = scaler.transform(feature_data.reshape(-1, 1)).reshape(feature_data.shape)
                    normalized_input[:, :, feature_idx] = torch.FloatTensor(normalized_feature)
            if self.target_col in self.scaler_dict:
                scaler = self.scaler_dict[self.target_col]
                normalized_target = scaler.transform(target.numpy().reshape(-1, 1)).flatten()
                normalized_target = torch.FloatTensor(normalized_target)
            else:
                normalized_target = target.clone()
            normalized_sequences.append((normalized_input, normalized_target))
        return normalized_sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]

    def get_scaler_dict(self):
        return self.scaler_dict if self.normalize else None

def create_data_loaders(df, batch_size, train_ratio=0.7, val_ratio=0.15):
    unique_days = sorted(df['Day'].unique())
    n_days = len(unique_days)
    train_days = unique_days[:int(n_days * train_ratio)]
    val_days = unique_days[int(n_days * train_ratio):int(n_days * (train_ratio + val_ratio))]
    test_days = unique_days[int(n_days * (train_ratio + val_ratio)):]
    train_df = df[df['Day'].isin(train_days)]
    val_df = df[df['Day'].isin(val_days)]
    test_df = df[df['Day'].isin(test_days)]
    train_dataset = WindTurbineDataset(train_df, normalize=False, fit_scaler=False)
    scaler_dict = train_dataset.get_scaler_dict()
    val_dataset = WindTurbineDataset(val_df, normalize=False, fit_scaler=False, scaler_dict=scaler_dict)
    test_dataset = WindTurbineDataset(test_df, normalize=False, fit_scaler=False, scaler_dict=scaler_dict)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, scaler_dict

train_loader, val_loader, test_loader, scalers = create_data_loaders(filtered_df1, batch_size=16)
for batch_idx, (inputs, targets) in enumerate(train_loader):
    break

In [6]:
torch.set_printoptions(sci_mode=False)
for batch_idx, (inputs, targets) in enumerate(train_loader):
    print(inputs[0, :, 0:2, :])
    print(targets[0, 0:2])
    print(inputs.shape)
    print(targets.shape)
    break

tensor([[[     3.5800,     -5.5700,     33.7800,     41.1200,     17.3600,
               0.9900,      0.9900,      0.9900,     -0.2100,    152.8300,
            3349.8516,   5939.2319,    410.0000,     81.0000],
         [     3.7700,     -0.9200,     28.8900,     36.4700,      4.2700,
               0.5100,      0.5100,      0.5100,    -25.4600,    146.5400,
            3351.0017,   6416.6470,    410.0000,     81.0000]]])
tensor([325.9900, 371.8200])
torch.Size([16, 1, 134, 14])
torch.Size([16, 134])


In [7]:
print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

969
261
263


In [ ]:
class TemporalGraphCreator:
    def __init__(self, epsilon=0.5):
        self.epsilon = epsilon

    def create_temporal_graphs(self, x):
        batch_size, timesteps, nodes, features = x.shape
        temporal_graphs = []
        for b in range(batch_size):
            batch_graphs = []
            for t in range(timesteps):
                node_features = x[b, t]
                dist_matrix = torch.cdist(node_features, node_features)
                edge_mask = (dist_matrix <= self.epsilon) & (dist_matrix > 0)
                edge_index = torch.nonzero(edge_mask).t()
                graph_data = Data(
                    x=node_features,
                    edge_index=edge_index,
                    num_nodes=nodes
                )
                batch_graphs.append(graph_data)
            temporal_graphs.append(batch_graphs)
        return temporal_graphs

In [8]:
class EncoderProcessorDecoder(nn.Module):
    def __init__(self, num_node_features, out_dim, hid_features=32): #64
        super(EncoderProcessorDecoder, self).__init__()
        self.hid_features = hid_features
        self.num_node_features = num_node_features
        self.out_dim = out_dim
        self.node_encoder = nn.Linear(num_node_features, hid_features, bias=True)
        self.gnn_processor = self._make_gnn()
        self.node_decoder = nn.Linear(hid_features, out_dim)

        self.fc = nn.Linear(out_dim, 1)

    def _make_gnn(self):
        convs = nn.ModuleList()
        for l in range(2):
            convs.append(GATConv(self.hid_features, self.hid_features, heads=1)) #GCN(in_channels=in_dim, in_channels, hidden_dim, out_channels=)
        return convs

    def forward(self, x):
        x = self.node_encoder(x)
        for i, conv in enumerate(self.gnn_processor):
            x = conv(x=x)
            x = torch.relu(x)
        x = self.node_decoder(x)
        x = torch.relu(x)

        x = self.fc(x)

        return x

In [20]:
"""
Encoder-processor-decoder with batch 16, timesteps=24, nodes=134, features=13
Xt = N x F -encoder> X1t -processor> X1l -decoder> Xt1 = N x F
X1 = self.relu(self.mlp_node_encoder(X))
X1l = self.gnn(St=create_temporal_graphs, X1, H common)
Xt1 = self.mlp_node_decoder(X1l)
"""

'\nXt = N x F\nX1t\nX1l\nXt1 = N x F\n'

In [11]:
model = EncoderProcessorDecoder(num_node_features=len(feature_cols), out_dim=len(feature_cols))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
model = model.to(device)
train_losses = []
for epoch in range(10):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # g = g.to(device)
        optimizer.zero_grad()

        roll_loss = []
        for i in range(timesteps):
            preds = model(inputs[:, i, :, :].to(device))
            loss = loss_fn(preds, targets.to(device))
            roll_loss.append(loss)
        loss1 = torch.stack(roll_loss).mean()

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch}: loss={total_loss/len(train_loader):.4f}")
torch.save(model.state_dict(), "/kaggle/working/attentive_gnn1.pth")

Using device: cpu


TypeError: GATConv.forward() missing 1 required positional argument: 'edge_index'

In [ ]:
# @torch.no_grad()
# def rollout_test(model, batch):
#     predicted_rollout = []
#     for time_step in range(final_step):
#         pred = model(temp)
#         predicted_rollout.append(pred)
#     return torch.stack(predicted_rollout, -1)

class LightningTrainer(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.batch_size = 12
        self.type_loss = trainer_options['type_loss']
        self.temporal_test_dataset_parameters = temporal_test_dataset_parameters
        self.rollout_steps = 1

    def training_step(self, batch):
        self.log("rollout_steps", torch.tensor(self.rollout_steps, dtype=torch.float32), on_step=False, on_epoch=True)
        roll_loss = []
        for i in range(self.rollout_steps):
            preds = self.model(temp)
            loss = loss_function(preds, temp.y[:, :, i])
            roll_loss.append(loss)
        loss = torch.stack(roll_loss).mean()
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = optim.Adam(lr=0.003,
                                weight_decay=0.7)
        lr_scheduler = optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                 step_size=20,
                                                 gamma=0.1)
        return [optimizer], [lr_scheduler]

    # def validation_step(self, batch):
    #     predicted_rollout = rollout_test(self.model, batch)
    #     #real_rollout = #
    #     #val_loss = #
    #     self.log("val_loss", val_loss, prog_bar=True)

    # def predict_step(self, batch):
    #     predicted_rollout = rollout_test(self.model, batch)
    #     return predicted_rollout

class DataModule(L.LightningDataModule):
    def __init__(self, temporal_train_dataset, temporal_val_dataset,
                 batch_size: int = 8):
        super().__init__()
        self.batch_size = batch_size
        self.temporal_train_dataset = temporal_train_dataset
        # self.temporal_val_dataset = temporal_val_dataset

    def train_dataloader(self):
        return DataLoader(self.temporal_train_dataset, batch_size=self.batch_size,
                          shuffle=True)

    # def val_dataloader(self):
    #     return DataLoader(self.temporal_val_dataset, batch_size=self.batch_size,
    #                       shuffle=False)

    def setup(self, stage=None):
        if stage == "fit" or stage is None:
            self.temporal_train_dataset = self.temporal_train_dataset
            # self.temporal_val_dataset = self.temporal_val_dataset

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, marker="o")
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/attentive_gnn.pth"))
model.to(device)
model.eval()
total_loss = 0
with torch.no_grad():
    for g in test_loader:
        g = g.to(device)
        pred = model(g.x, W_tensor)
        loss = loss_fn(pred.squeeze(), g.y)
        total_loss += loss.item()
avg_loss = total_loss / len(test_loader)
print(f"Test Loss: {avg_loss:.4f}")

In [ ]:
model.eval()
preds, trues = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_pred = model(X_batch)
        preds.extend(y_pred.numpy())
        trues.extend(y_batch.numpy())
preds = np.array(preds)
trues = np.array(trues)
mse = np.mean((preds - trues) ** 2)
print("Test MSE:", mse)

In [ ]:
step = 250
plt.figure(figsize=(12, 6))
plt.plot(trues[::step], label="True Patv", alpha=0.7)
plt.plot(preds[::step], label="Predicted Patv", alpha=0.7)
plt.title("LSTM Forecast: True vs Predicted Patv (Downsampled)")
plt.xlabel("Sample")
plt.ylabel("Patv")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()